# Chandigarh University - MBA Analytics Major Project
## Student Performance & Placement Analytics
**Problem Statement**: Business schools lack insight into how academic scores, attendance, and specialisation affect students' placement outcomes.
**Objective**: Analyze student marks, attendance, specialisation, and placement data; calculate grades and performance bands; evaluate econometric links with placement CTC; and generate prescriptive academic interventions.

### 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, classification_report, roc_auc_score

# Style configuration
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('students_dataset.csv')
print(f'Cohort Dimensions: {df.shape[0]} students, {df.shape[1]} attributes')
df.head()

### 2. Descriptive Analytics & Academic Grading Distribution

In [ ]:
# Overall descriptive statistics
display_cols = ['overall_cgpa', 'attendance_percentage', 'aptitude_score', 'soft_skills_score', 'certifications_count', 'placement_salary_lpa']
df[display_cols].describe().round(2)

In [ ]:
# Performance Bands Breakdown
band_dist = df['performance_band'].value_counts(normalize=True) * 100
print('Performance Bands (%):\n', band_dist.round(1))

# Grade Distribution
grade_order = ['O', 'A+', 'A', 'B+', 'B', 'C']
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='grade', order=grade_order, palette='mako')
plt.title('Grade Distribution Across MBA Cohort (10-Point Scale)')
plt.xlabel('Academic Grade')
plt.ylabel('Student Count')
plt.show()

### 3. Hypothesis Testing: The 75% Attendance Threshold Effect
Does meeting university attendance threshold (>=75%) significantly impact academic CGPA and placement package?

In [ ]:
low_att = df[df['attendance_percentage'] < 75.0]
high_att = df[df['attendance_percentage'] >= 75.0]

t_stat_cgpa, p_val_cgpa = stats.ttest_ind(high_att['overall_cgpa'], low_att['overall_cgpa'], equal_var=False)
print(f'CGPA Comparison:')
print(f'  - Attendance < 75% Mean CGPA: {low_att["overall_cgpa"].mean():.2f}')
print(f'  - Attendance >= 75% Mean CGPA: {high_att["overall_cgpa"].mean():.2f}')
print(f'  - Welch t-statistic: {t_stat_cgpa:.3f}, p-value: {p_val_cgpa:.4e}')

print('\nPlacement Rate Comparison:')
print(f'  - Low Attendance (<75%) Placement Rate: {(low_att["placement_status"] == "Placed").mean()*100:.1f}%')
print(f'  - High Attendance (>=75%) Placement Rate: {(high_att["placement_status"] == "Placed").mean()*100:.1f}%')

### 4. Correlation Analysis: Academic Metrics vs Placement CTC

In [ ]:
placed_df = df[df['placement_status'] == 'Placed']
corr_subset = placed_df[['overall_cgpa', 'attendance_percentage', 'aptitude_score', 'soft_skills_score', 'certifications_count', 'placement_salary_lpa']]
plt.figure(figsize=(9, 7))
sns.heatmap(corr_subset.corr(), annot=True, fmt='.2f', cmap='Blues', square=True)
plt.title('Correlation Matrix (Placed MBA Candidates)')
plt.show()

### 5. Econometric Modeling: Multiple Linear Regression for Placement CTC

In [ ]:
# Feature Matrix
X_reg = pd.get_dummies(placed_df[['overall_cgpa', 'attendance_percentage', 'aptitude_score', 'soft_skills_score', 'certifications_count', 'live_projects_count', 'specialisation']], drop_first=True)
y_reg = placed_df['placement_salary_lpa']

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)
reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)
print(f'Multiple Linear Regression Performance:')
print(f'  - R² Score: {r2_score(y_test, y_pred):.3f}')
print(f'  - Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.3f} LPA')

# Regression Coefficients
coef_df = pd.DataFrame({'Feature': X_reg.columns, 'Coefficient (LPA Impact)': reg.coef_})
coef_df = coef_df.sort_values(by='Coefficient (LPA Impact)', ascending=False)
display(coef_df)

### 6. Prescriptive Interventions: Academic Remediation Matrix

In [ ]:
print('Academic Interventions Distribution:')
print(df['intervention_category'].value_counts())

# Export priority at-risk list
at_risk = df[df['performance_band'].isin(['Band 3: Skill Deficit (Moderate Risk)', 'Band 4: Critical Academic Intervention'])]
print(f'\nTotal Students Flagged for Intervention: {len(at_risk)}')
at_risk[['student_id', 'full_name', 'specialisation', 'overall_cgpa', 'attendance_percentage', 'recommended_intervention']].head(10)